# Online shop orders: how customers pay

**Frequency Distribution for Categorical Data** · a worked example from the StatNex teaching guide

**The story.** An online shop looked at its last 200 orders to see how customers pay. Before adding a new payment option or dropping an old one, the team wants the basic picture.

**The question.** How do our orders split across payment methods?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `04_crosstab/categorical_frequency.csv` (200 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Payment method
1.0,Mobile wallet
2.0,Credit card
3.0,Mobile wallet
4.0,Mobile wallet
5.0,Mobile wallet
6.0,Cash on delivery
7.0,Credit card
8.0,Mobile wallet
9.0,Cash on delivery
10.0,Mobile wallet
11.0,Credit card
12.0,Cash on delivery
13.0,Credit card
14.0,Cash on delivery
15.0,Cash on delivery
16.0,Cash on delivery
17.0,Mobile wallet
18.0,Cash on delivery
19.0,Cash on delivery
20.0,Mobile wallet
21.0,Credit card
22.0,Mobile wallet
23.0,Mobile wallet
24.0,Mobile wallet
25.0,Mobile wallet
26.0,Credit card
27.0,Credit card
28.0,Credit card
29.0,Mobile wallet
30.0,Credit card
31.0,Mobile wallet
32.0,Cash on delivery
33.0,Cash on delivery
34.0,Cash on delivery
35.0,Cash on delivery
36.0,Credit card
37.0,Cash on delivery
38.0,Mobile wallet
39.0,Mobile wallet
40.0,Credit card
41.0,Mobile wallet
42.0,Credit card
43.0,Mobile wallet
44.0,Credit card
45.0,Credit card
46.0,Credit card
47.0,Mobile wallet
48.0,Credit card
49.0,Credit card
50.0,Mobile wallet
51.0,Credit card
52.0,Cash on delivery
53.0,Credit card
54.0,Credit card
55.0,Cash on delivery
56.0,Credit card
57.0,Credit card
58.0,Credit card
59.0,Cash on delivery
60.0,Credit card
61.0,Credit card
62.0,Credit card
63.0,Mobile wallet
64.0,Cash on delivery
65.0,Credit card
66.0,Credit card
67.0,Mobile wallet
68.0,Credit card
69.0,Mobile wallet
70.0,Credit card
71.0,Mobile wallet
72.0,Credit card
73.0,Mobile wallet
74.0,Credit card
75.0,Credit card
76.0,Credit card
77.0,Mobile wallet
78.0,Credit card
79.0,Credit card
80.0,Cash on delivery
81.0,Credit card
82.0,Mobile wallet
83.0,Mobile wallet
84.0,Cash on delivery
85.0,Mobile wallet
86.0,Cash on delivery
87.0,Credit card
88.0,Cash on delivery
89.0,Cash on delivery
90.0,Cash on delivery
91.0,Mobile wallet
92.0,Credit card
93.0,Mobile wallet
94.0,Mobile wallet
95.0,Cash on delivery
96.0,Cash on delivery
97.0,Credit card
98.0,Mobile wallet
99.0,Credit card
100.0,Credit card
101.0,Credit card
102.0,Credit card
103.0,Credit card
104.0,Cash on delivery
105.0,Mobile wallet
106.0,Credit card
107.0,Credit card
108.0,Mobile wallet
109.0,Mobile wallet
110.0,Mobile wallet
111.0,Mobile wallet
112.0,Mobile wallet
113.0,Credit card
114.0,Credit card
115.0,Cash on delivery
116.0,Credit card
117.0,Credit card
118.0,Credit card
119.0,Credit card
120.0,Mobile wallet
121.0,Mobile wallet
122.0,Credit card
123.0,Cash on delivery
124.0,Credit card
125.0,Credit card
126.0,Credit card
127.0,Credit card
128.0,Credit card
129.0,Cash on delivery
130.0,Mobile wallet
131.0,Credit card
132.0,Credit card
133.0,Mobile wallet
134.0,Credit card
135.0,Credit card
136.0,Mobile wallet
137.0,Mobile wallet
138.0,Mobile wallet
139.0,Credit card
140.0,Credit card
141.0,Mobile wallet
142.0,Cash on delivery
143.0,Cash on delivery
144.0,Mobile wallet
145.0,Mobile wallet
146.0,Mobile wallet
147.0,Cash on delivery
148.0,Mobile wallet
149.0,Cash on delivery
150.0,Cash on delivery
151.0,Credit card
152.0,Credit card
153.0,Mobile wallet
154.0,Cash on delivery
155.0,Credit card
156.0,Cash on delivery
157.0,Mobile wallet
158.0,Mobile wallet
159.0,Cash on delivery
160.0,Cash on delivery
161.0,Credit card
162.0,Credit card
163.0,Credit card
164.0,Mobile wallet
165.0,Mobile wallet
166.0,Cash on delivery
167.0,Cash on delivery
168.0,Credit card
169.0,Credit card
170.0,Credit card
171.0,Mobile wallet
172.0,Credit card
173.0,Credit card
174.0,Cash on delivery
175.0,Mobile wallet
176.0,Mobile wallet
177.0,Mobile wallet
178.0,Credit card
179.0,Mobile wallet
180.0,Mobile wallet
181.0,Mobile wallet
182.0,Cash on delivery
183.0,Mobile wallet
184.0,Credit card
185.0,Cash on delivery
186.0,Credit card
187.0,Mobile wallet
188.0,Credit card
189.0,Credit card
190.0,Credit card
191.0,Credit card
192.0,Mobile wallet
193.0,Mobile wallet
194.0,Mobile wallet
195.0,Mobile wallet
196.0,Cash on delivery
197.0,Cash on delivery
198.0,Mobile wallet
199.0,Mobile wallet
200.0,Credit card

## Libraries

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.width", 120)


def label(value):
    """A category's name as the app prints it: 2.0 read from a number column is "2"."""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def cp932(name):
    """Sort key: the name's Shift-JIS (CP932) bytes, the order a Japanese Excel uses.

    For plain English text this is ordinary ASCII order (capitals before small letters).
    """
    try:
        return name.encode("cp932")
    except UnicodeEncodeError:
        return name.encode("utf-8")

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Payment method"]
# One frequency table per column.
series = {name: data[name] for name in columns}

## Frequency tables

In [ ]:
tables = []
for name, values in series.items():
    # Blank cells are not a category: they are left out, and n counts what remains.
    values = values[values.notna() & (values.astype(str).str.strip() != "")]
    labels = [label(value) for value in values]
    counts = Counter(labels)  # keeps the order in which categories first appear
    # By name: numeric codes by value (1, 2, 10), other names in Shift-JIS byte order
    # (the order a Japanese Excel sorts them in). Blank cells, if kept, come last.
    named = [category for category in counts if category != "(blank)"]
    try:
        names = sorted(named, key=float)
    except ValueError:
        names = sorted(named, key=cp932)
    names += [category for category in counts if category == "(blank)"]
    n = len(labels)
    table = pd.DataFrame({"frequency": [counts[category] for category in names]}, index=names)
    table["relative frequency"] = table["frequency"] / n
    table["cumulative relative frequency"] = table["relative frequency"].cumsum()
    tables.append(table)
    print(f"\n{name}")
    print(table)
    print(f"categories = {len(names)}, n = {n}")

## Chart (of the first table)

In [ ]:
Path("figures").mkdir(exist_ok=True)
first = tables[0]
figure, axis = plt.subplots(figsize=(6, 4))
axis.pie(first["frequency"], labels=first.index, autopct="%.1f%%", counterclock=False, startangle=90)
figure.tight_layout()
figure.savefig("figures/chart.png", dpi=120)

## What the result says

A frequency table counts how many cases fall in each category. Of the 200 orders, 84 were paid by credit card, 71 by mobile wallet and 45 by cash on delivery.

The relative frequency is the share of all orders in one category. Credit card has the largest share, 0.42, which is 42% of orders. Mobile wallet has 35.5% and cash on delivery 22.5%.

So credit card is the most common way to pay, but it is not the majority: the other two methods together cover more than half of the orders. Even the smallest method, cash on delivery, pays for more than one order in five. The team should not drop any of the three without a closer look.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.